# Global Development Lakehouse — Dashboard Queries

Exploratory and business-facing queries against the Gold layer, answering 
key questions about the relationship between education investment, 
population trends, and GDP growth across countries and time. These queries 
are validated here first, then recreated in Databricks SQL Editor to power 
the final dashboard.

**Source table:** `workspace.global_development.gold_country_development_metrics`  
**Coverage tables (for data completeness questions):** `silver_*_with_nulls`

In [0]:
#Load the Gold table once, reuse throughout
df_gold = spark.table("workspace.global_development.gold_country_development_metrics")

# Register as a temp view so we can also use pure SQL cells/spark.sql() interchangeably
df_gold.createOrReplaceTempView("gold_metrics")

In [0]:
# Load the _with_nulls tables (needed for coverage questions later)
df_gdp_with_nulls = spark.table("workspace.global_development.silver_gdp_growth_with_nulls")
df_population_with_nulls = spark.table("workspace.global_development.silver_population_with_nulls")
df_education_with_nulls = spark.table("workspace.global_development.silver_education_with_nulls")

df_gdp_with_nulls.createOrReplaceTempView("gdp_with_nulls")
df_population_with_nulls.createOrReplaceTempView("population_with_nulls")
df_education_with_nulls.createOrReplaceTempView("education_with_nulls")

### Q1: Does government education investment correlate with GDP growth in the same year?

In [0]:
'''Approach: Compute the Pearson correlation coefficient between 
`education_expenditure_pct_gdp` and `gdp_growth_pct` across all country-years 
in the Gold table. Correlation ranges from -1 (strong inverse) to +1 (strong 
positive); values near 0 indicate a weak or no linear relationship.'''

'Approach: Compute the Pearson correlation coefficient between \n`education_expenditure_pct_gdp` and `gdp_growth_pct` across all country-years \nin the Gold table. Correlation ranges from -1 (strong inverse) to +1 (strong \npositive); values near 0 indicate a weak or no linear relationship.'

In [0]:
result = spark.sql("""
    SELECT ROUND(corr(education_expenditure_pct_gdp, gdp_growth_pct), 3) AS correlation
    FROM gold_metrics
    WHERE education_expenditure_pct_gdp IS NOT NULL 
      AND gdp_growth_pct IS NOT NULL
""")

result.show()

+-----------+
|correlation|
+-----------+
|      -0.09|
+-----------+



***Finding: Same-year correlation between education spending and GDP growth 
is very weak (-0.09), suggesting no meaningful immediate relationship. This 
is consistent with economic theory — education investment typically affects 
productivity and growth with a multi-year lag, not instantaneously. This 
motivates the next analysis: testing correlation with a time lag.***

### Q2: Education-GDP correlation restricted to 1990-2025, overall and by region

Approach: Recompute the Pearson correlation between education spending and 
GDP growth, limited to 1990-2025 (more complete, more relevant modern data), 
and broken down by region using GROUPING SETS to see both the overall figure 
and each region's figure in a single query. This tests whether the weak 
same-year correlation seen in Q1 holds globally, or whether specific regions 
show a stronger relationship.

In [0]:
spark.sql("""
    SELECT DISTINCT Region 
    FROM gold_metrics 
    WHERE Region IS NOT NULL
    ORDER BY Region
""").show(30, truncate=False)

+--------------------------+
|Region                    |
+--------------------------+
|East Asia & Pacific       |
|Europe & Central Asia     |
|Latin America & Caribbean |
|Middle East & North Africa|
|North America             |
|South Asia                |
|Sub-Saharan Africa        |
+--------------------------+



In [0]:
combined_result = spark.sql("""
    SELECT 
        COALESCE(Region, 'ALL REGIONS (Overall)') AS Region,
        ROUND(corr(education_expenditure_pct_gdp, gdp_growth_pct), 3) AS correlation,
        COUNT(*) AS num_records
    FROM gold_metrics
    WHERE education_expenditure_pct_gdp IS NOT NULL 
      AND gdp_growth_pct IS NOT NULL
      AND year BETWEEN 1990 AND 2025
      AND Region IS NOT NULL
    GROUP BY GROUPING SETS ( (Region), () )
    ORDER BY correlation DESC
""")

combined_result.show(20, truncate=False)

+--------------------------+-----------+-----------+
|Region                    |correlation|num_records|
+--------------------------+-----------+-----------+
|North America             |0.279      |56         |
|Middle East & North Africa|0.188      |356        |
|Sub-Saharan Africa        |-0.016     |950        |
|South Asia                |-0.031     |139        |
|ALL REGIONS (Overall)     |-0.102     |4020       |
|Latin America & Caribbean |-0.11      |773        |
|Europe & Central Asia     |-0.181     |1216       |
|East Asia & Pacific       |-0.184     |530        |
+--------------------------+-----------+-----------+



***Finding: The education-GDP relationship is not uniform globally — it 
varies by region and even changes direction. North America (+0.28) and 
MENA (+0.19) show a positive relationship, while Europe & Central Asia 
(-0.18) and East Asia & Pacific (-0.18) show a negative one. Sub-Saharan 
Africa and South Asia show virtually no linear relationship. This suggests 
regional economic structure and government spending patterns (e.g., 
counter-cyclical education spending during downturns) significantly 
shape this relationship, and a single global correlation figure obscures 
this nuance. Note: North America's result is based on a small sample 
(56 records) and should be interpreted cautiously.***

### Q3: Decade-by-decade global trends — GDP growth, population growth, and education spending

Approach: Aggregate the Gold table by decade (already available as a derived 
column), computing average GDP growth, average population growth, and 
average education spending globally for each decade from the 1960s through 
2020s. This shows how each metric has trended over time and sets up a 
clearer "past vs. present" comparison before testing lagged effects.

In [0]:
decade_trends = spark.sql("""
    SELECT 
        decade,
        ROUND(AVG(gdp_growth_pct), 2) AS avg_gdp_growth,
        ROUND(AVG(population_growth_rate), 2) AS avg_population_growth,
        ROUND(AVG(education_expenditure_pct_gdp), 2) AS avg_education_spend,
        COUNT(DISTINCT Country_Code) AS countries_reporting
    FROM gold_metrics
    WHERE decade IS NOT NULL
    GROUP BY decade
    ORDER BY decade
""")

decade_trends.show(20, truncate=False)

+------+--------------+---------------------+-------------------+-------------------+
|decade|avg_gdp_growth|avg_population_growth|avg_education_spend|countries_reporting|
+------+--------------+---------------------+-------------------+-------------------+
|1960  |5.43          |2.35                 |NULL               |216                |
|1970  |5.02          |2.17                 |4.14               |216                |
|1980  |3.0           |2.09                 |4.08               |216                |
|1990  |2.99          |1.63                 |4.16               |217                |
|2000  |4.02          |1.5                  |4.39               |217                |
|2010  |3.18          |1.27                 |4.34               |217                |
|2020  |2.46          |1.02                 |4.39               |217                |
+------+--------------+---------------------+-------------------+-------------------+



### Q4: Relationship between all three metrics — population growth, education spending, and GDP growth

Approach: Compute pairwise correlations between all three metrics 
(population growth ↔ GDP growth, population growth ↔ education spending, 
education spending ↔ GDP growth), globally and by decade. This reveals 
whether these three factors tend to move together (e.g., "developed 
economy" pattern: population growth slows, education spend rises, GDP 
growth moderates) or independently.

In [0]:
combined_trends = spark.sql("""
    SELECT 
        decade,
        ROUND(AVG(gdp_growth_pct), 2) AS avg_gdp_growth,
        ROUND(AVG(population_growth_rate), 2) AS avg_population_growth,
        ROUND(AVG(education_expenditure_pct_gdp), 2) AS avg_education_spend,
        COUNT(DISTINCT Country_Code) AS countries_reporting,
        ROUND(corr(population_growth_rate, gdp_growth_pct), 3) AS corr_population_vs_gdp,
        ROUND(corr(population_growth_rate, education_expenditure_pct_gdp), 3) AS corr_population_vs_education,
        ROUND(corr(education_expenditure_pct_gdp, gdp_growth_pct), 3) AS corr_education_vs_gdp
    FROM gold_metrics
    WHERE decade IS NOT NULL
    GROUP BY decade
    ORDER BY decade
""")

combined_trends.show(20, truncate=False)

+------+--------------+---------------------+-------------------+-------------------+----------------------+----------------------------+---------------------+
|decade|avg_gdp_growth|avg_population_growth|avg_education_spend|countries_reporting|corr_population_vs_gdp|corr_population_vs_education|corr_education_vs_gdp|
+------+--------------+---------------------+-------------------+-------------------+----------------------+----------------------------+---------------------+
|1960  |5.43          |2.35                 |NULL               |216                |0.108                 |NULL                        |NULL                 |
|1970  |5.02          |2.17                 |4.14               |216                |0.216                 |-0.067                      |-0.077               |
|1980  |3.0           |2.09                 |4.08               |216                |0.007                 |-0.127                      |-0.015               |
|1990  |2.99          |1.63             

***Finding: Three clear multi-decade trends emerge: (1) population growth 
has declined steadily and consistently, from 2.35% in the 1960s to 1.02% 
in the 2020s — a well-documented global demographic transition; (2) 
education spending has gradually risen over the same period, from ~4.1% 
to ~4.4% of GDP; (3) GDP growth has generally slowed since the high-growth 
post-war decades. Notably, population growth and education spending show a 
consistent negative correlation in every decade (-0.07 to -0.21), 
supporting the "quantity-quality tradeoff" theory in development economics: 
as population growth slows, countries tend to invest more per capita in 
education. Education spending's relationship with GDP growth remains weak 
throughout, reinforcing that its economic effect — if any — is likely 
delayed rather than immediate.***

###Q5: Does education investment show a delayed effect on GDP growth (5-year and 10-year lag)?

Approach: For each country, shift GDP growth values backward in time using 
a window function, so we can compare this year's education spending against 
GDP growth 5 and 10 years later — rather than assuming an instant, same-year 
effect. This directly tests the hypothesis from above questions that education's 
economic payoff is delayed, not immediate.

In [0]:
from pyspark.sql import Window
import pyspark.sql.functions as F

country_window = Window.partitionBy("Country_Code").orderBy("year")

df_lagged = (
    df_gold
    .withColumn("gdp_growth_5yr_later", F.lead("gdp_growth_pct", 5).over(country_window))
    .withColumn("gdp_growth_10yr_later", F.lead("gdp_growth_pct", 10).over(country_window))
)

df_lagged.createOrReplaceTempView("gold_lagged")

result_lagged = spark.sql("""
    SELECT 
        ROUND(corr(education_expenditure_pct_gdp, gdp_growth_pct), 3) AS same_year_corr,
        ROUND(corr(education_expenditure_pct_gdp, gdp_growth_5yr_later), 3) AS lag_5yr_corr,
        ROUND(corr(education_expenditure_pct_gdp, gdp_growth_10yr_later), 3) AS lag_10yr_corr
    FROM gold_lagged
    WHERE education_expenditure_pct_gdp IS NOT NULL
""")

result_lagged.show(truncate=False)

+--------------+------------+-------------+
|same_year_corr|lag_5yr_corr|lag_10yr_corr|
+--------------+------------+-------------+
|-0.09         |-0.067      |-0.07        |
+--------------+------------+-------------+



***Finding: Introducing a 5-year or 10-year lag does not meaningfully 
strengthen the education-GDP correlation (-0.09 same-year vs. -0.067 to 
-0.07 lagged) — the relationship remains weak regardless of delay. This 
suggests either (a) a simple linear year-over-year correlation cannot 
capture education's cumulative/non-linear effect on growth, (b) other 
short-term economic forces (trade, commodity prices, financial cycles) 
dominate GDP growth variance far more than education spending, or (c) 
spending levels alone don't capture spending *effectiveness*, which is 
what likely matters more for growth outcomes. Combined with the regional 
findings (Q1C), this suggests the education-GDP relationship is shaped 
more by regional economic structure than by simple time delay.***

### Q6: Which regions have the biggest gaps in education spending data reporting?

Approach: Using the `education_with_nulls` table (which preserves every 
country-year combination, including missing data), calculate what 
percentage of expected records are actually missing, grouped by region. 
This reveals which parts of the world have the least reliable/complete 
World Bank reporting for this indicator — a data-quality insight, not an 
economic one.

In [0]:
coverage_by_region = spark.sql("""
    SELECT 
        m.Region,
        COUNT(*) AS total_country_years,
        SUM(CASE WHEN e.education_expenditure_pct_gdp IS NULL THEN 1 ELSE 0 END) AS missing_records,
        ROUND(SUM(CASE WHEN e.education_expenditure_pct_gdp IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_missing
    FROM education_with_nulls e
    JOIN workspace.global_development.bronze_country_metadata m
        ON e.Country_Code = m.Country_Code
    WHERE m.Region IS NOT NULL
    GROUP BY m.Region
    ORDER BY pct_missing DESC
""")

coverage_by_region.show(20, truncate=False)

+--------------------------+-------------------+---------------+-----------+
|Region                    |total_country_years|missing_records|pct_missing|
+--------------------------+-------------------+---------------+-----------+
|East Asia & Pacific       |2442               |1741           |71.3       |
|Latin America & Caribbean |2772               |1835           |66.2       |
|Middle East & North Africa|1518               |971            |64.0       |
|Sub-Saharan Africa        |3168               |1985           |62.7       |
|North America             |198                |120            |60.6       |
|Europe & Central Asia     |3828               |2315           |60.5       |
|South Asia                |396                |230            |58.1       |
+--------------------------+-------------------+---------------+-----------+



***Finding: Education spending data has substantial reporting gaps 
globally — every region is missing 58-71% of expected country-year 
records. This is important context for the earlier correlation findings 
above questions, those results were computed on a minority subset of all 
possible data points, which may not be representative. East Asia & Pacific 
shows the highest gap (71.3%), though this likely reflects the region's 
inclusion of many small Pacific Island nations with limited statistical 
infrastructure, rather than poor reporting by the region's larger 
economies. This finding is a useful caveat on data reliability, not a 
judgment on any single country or region's institutional quality.***

### Q7: Which individual countries show the strongest positive relationship between education spending and GDP growth?

Approach: Compute per-country correlation between education spending and 
GDP growth (requires enough data points per country to be meaningful), 
ranked to surface specific success stories or outliers — moving from 
regional-level patterns down to concrete, nameable examples for the 
dashboard narrative.

In [0]:
country_correlation = spark.sql("""
    SELECT 
        Country_Name,
        Region,
        ROUND(corr(education_expenditure_pct_gdp, gdp_growth_pct), 3) AS correlation,
        COUNT(*) AS num_years
    FROM gold_metrics
    WHERE education_expenditure_pct_gdp IS NOT NULL 
      AND gdp_growth_pct IS NOT NULL
    GROUP BY Country_Name, Region
    HAVING COUNT(*) >= 15
    ORDER BY correlation DESC
    LIMIT 20
""")

country_correlation.show(20, truncate=False)

+------------------------------+--------------------------+-----------+---------+
|Country_Name                  |Region                    |correlation|num_years|
+------------------------------+--------------------------+-----------+---------+
|Kuwait                        |Middle East & North Africa|0.753      |33       |
|Puerto Rico (US)              |Latin America & Caribbean |0.702      |20       |
|Belarus                       |Europe & Central Asia     |0.646      |22       |
|Japan                         |East Asia & Pacific       |0.615      |42       |
|Dominica                      |Latin America & Caribbean |0.535      |16       |
|Ethiopia                      |Sub-Saharan Africa        |0.518      |37       |
|Haiti                         |Latin America & Caribbean |0.511      |16       |
|Tanzania                      |Sub-Saharan Africa        |0.458      |31       |
|Lao PDR                       |East Asia & Pacific       |0.449      |26       |
|Guinea         

***Finding: Kuwait (0.75) and Japan (0.62) show the strongest, most 
reliable positive relationships between education spending and GDP growth, 
each backed by 30+ years of data. Notably, several Sub-Saharan African 
countries (Ethiopia, Tanzania, Ghana) also show moderate positive 
correlations (0.42-0.52), despite the region as a whole showing almost no 
relationship (Q1C: -0.016) — indicating the regional aggregate obscures 
real country-level variation. Smaller nations in this list (Dominica, 
Haiti) should be interpreted cautiously given smaller sample sizes and 
higher economic volatility.***

### Q8: Which countries are the biggest "outliers" — high GDP growth with low education investment, or high education investment with low GDP growth?

Approach: Compare each country's average education spending and average 
GDP growth (over a recent period, e.g. 2000-2025) against global averages, 
identifying countries that defy the expected pattern — useful for 
highlighting interesting individual cases beyond the aggregate trends.

In [0]:
outliers = spark.sql("""
    WITH country_avgs AS (
        SELECT 
            Country_Name,
            Region,
            ROUND(AVG(gdp_growth_pct), 2) AS avg_gdp_growth,
            ROUND(AVG(education_expenditure_pct_gdp), 2) AS avg_education_spend,
            COUNT(*) AS num_years
        FROM gold_metrics
        WHERE year BETWEEN 2000 AND 2025
          AND gdp_growth_pct IS NOT NULL
          AND education_expenditure_pct_gdp IS NOT NULL
        GROUP BY Country_Name, Region
        HAVING COUNT(*) >= 10
    ),
    global_avgs AS (
        SELECT 
            ROUND(AVG(avg_gdp_growth), 2) AS global_avg_gdp,
            ROUND(AVG(avg_education_spend), 2) AS global_avg_education
        FROM country_avgs
    )
    SELECT 
        c.Country_Name,
        c.Region,
        c.avg_gdp_growth,
        c.avg_education_spend,
        g.global_avg_gdp,
        g.global_avg_education,
        ROUND(c.avg_gdp_growth - g.global_avg_gdp, 2) AS gdp_vs_global,
        ROUND(c.avg_education_spend - g.global_avg_education, 2) AS education_vs_global
    FROM country_avgs c
    CROSS JOIN global_avgs g
    ORDER BY gdp_vs_global DESC
    LIMIT 10
""")

outliers.show(20, truncate=False)

+------------------------+--------------------------+--------------+-------------------+--------------+--------------------+-------------+-------------------+
|Country_Name            |Region                    |avg_gdp_growth|avg_education_spend|global_avg_gdp|global_avg_education|gdp_vs_global|education_vs_global|
+------------------------+--------------------------+--------------+-------------------+--------------+--------------------+-------------+-------------------+
|Turks and Caicos Islands|Latin America & Caribbean |10.42         |2.78               |3.43          |4.39                |6.99         |-1.61              |
|Ethiopia                |Sub-Saharan Africa        |8.49          |4.49               |3.43          |4.39                |5.06         |0.1                |
|Tajikistan              |Europe & Central Asia     |7.66          |4.2                |3.43          |4.39                |4.23         |-0.19              |
|Azerbaijan              |Europe & Central Asi

***Finding: The 10 countries with the highest average GDP growth 
(2000-2025) mostly show BELOW-average education spending, not above — 
e.g., Cambodia (+3.03 growth, -2.74 education spend), Turks and Caicos 
(+6.99 growth, -1.61 education spend). This does not suggest education 
spending harms growth; rather, these countries' growth is driven by other 
factors — resource booms (Azerbaijan), specialized economies (Macao, 
Qatar), or "catch-up" convergence growth typical of lower-income economies 
starting from a small base. This provides concrete, country-level context 
for the weak education-GDP correlation found in Q1/Q2: high growth in this 
dataset is more often explained by structural economic factors than by 
education investment levels.***